# 📈 Modul 05: Supervised Learning II - Regresi (*Regression*)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/05_Regression_Algorithms.ipynb)
[![Course](https://img.shields.io/badge/Course-Data%20Mining%20Zero%20to%20Hero-blue.svg)](https://github.com/antonprafanto/data-mining-zero-to-hero)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

> **Pengampu**: Anton Prafanto  
> **Prinsip Pembelajaran**: *Intuition before Formulas* — Pahami filosofi & analogi dunia nyata sebelum rumus matematika dan kode.

---

### 🗺️ Peta Posisi Modul dalam CRISP-DM
```
[1. Business Understanding] ➔ [2. Data Understanding] ➔ [3. Data Preparation]
                                                               │
                                                               ▼
[6. Deployment] 💡 ◄── [5. Evaluation] ◄── 🎯 [4. MODELING: REGRESI]
```
Jika di **Modul 04** kita memprediksi **kategori** (misal: Sakit vs Sehat), di **Modul 05** kita melangkah ke tugas krusial berikutnya: **Memprediksi nilai angka kontinu (*Continuous Value Prediction*)** seperti harga rumah, omzet penjualan, inflasi, atau suhu cuaca!


## 🏡 1. Filosofi & Intuisi Regresi: Menaksir Angka Masa Depan

### 💼 Analogi Dunia Nyata: Taksiran Makelar Properti
Bayangkan Anda bertanya kepada seorang agen properti berpengalaman: *"Berapa perkiraan harga rumah di daerah ini?"*  
Makelar tersebut akan menjawab:
1. *"Harga tanah dasarnya sekitar **Rp 200 juta** (ini disebut **Titik Potong / Intercept $\beta_0$**)."*
2. *"Setiap tambahan **1 meter persegi luas bangunan**, harganya bertambah sekitar **Rp 5 juta** (ini disebut **Kemiringan / Slope $\beta_1$**)."*
3. *"Jika ada garasi mobil, tambahkan lagi **Rp 30 juta** (koefisien $\beta_2$). Namun jika usia bangunan sudah tua 10 tahun, kurangi **Rp 15 juta** (koefisien $\beta_3$)."*

Secara intuitif, pikiran makelar tersebut bekerja persis seperti rumus **Multiple Linear Regression**:
$$\hat{Y} = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_p X_p$$

### 📐 Bagaimana Komputer Menemukan Nilai $\beta$? (Metode OLS)
Komputer mencari garis lurus terbaik yang menembus kumpulan titik data dengan metode **Ordinary Least Squares (OLS)**. Tujuannya sederhana: **Meminimalkan jumlah kuadrat jarak kesalahan / sisa (*Sum of Squared Residuals*)** antara tebakan garis dengan titik kenyataan asli!


## 📂 2. Memuat Dataset Nyata: California Housing & Partisi Data

Kita akan menggunakan dataset standar industri: **California Housing Dataset** dari Scikit-Learn.  
Dataset ini memuat data sensus perumahan di California dengan fitur-fitur seperti:
* `MedInc` : Pendapatan median warga di area tersebut (dalam puluhan ribu USD).
* `HouseAge` : Usia rata-rata bangunan rumah.
* `AveRooms` : Rata-rata jumlah kamar tidur/ruangan per rumah.
* `Population` : Jumlah penduduk di blok sensus tersebut.
* Target `MedHouseVal` : Nilai median harga rumah (dalam ratusan ribu USD, misal 2.0 = $200.000).

> 💡 **Catatan Penting Partisi Data Regresi**:  
> Berbeda dengan Klasifikasi di Modul 04, pada Regresi kita **tidak menggunakan parameter `stratify=y`**, karena nilai target berbentuk angka kontinu tak terbatas, bukan kategori diskrit berlabel terpisah!


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Memuat dataset California Housing resmi dari Scikit-Learn
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(f"Dimensi Dataset: {df.shape[0]} baris × {df.shape[1]} kolom")
print("\n5 Baris Teratas:")
df.head()


In [ ]:
# Pisahkan Fitur (X) dan Target (y)
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

# Partisi Data: 80% Train Set (Data Latih), 20% Test Set (Data Uji)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standarisasi Fitur
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Data Latih (Train Set): {X_train.shape[0]} baris")
print(f"Data Uji   (Test Set) : {X_test.shape[0]} baris")


## 🔍 3. Eksplorasi Regresi Linear Sederhana (*Simple Linear Regression*)

Sebelum menggunakan seluruh 8 fitur, mari kita uji intuisi kita menggunakan **1 fitur paling berpengaruh**:  
Apakah orang berpendapatan tinggi tinggal di rumah yang lebih mahal? Mari amati hubungan antara `MedInc` (Pendapatan) dan `MedHouseVal` (Harga Rumah).


In [ ]:
from sklearn.linear_model import LinearRegression

# Mengambil 1 fitur saja (MedInc: Median Income)
X_train_single = X_train[['MedInc']]
X_test_single = X_test[['MedInc']]

# Latih Model Simple Linear Regression
slr = LinearRegression()
slr.fit(X_train_single, y_train)

# Ekstraksi Parameter Garis: y = b0 + b1 * x
b0 = slr.intercept_
b1 = slr.coef_[0]

print("=== PERSAMAAN GARIS REGRESI SEDERHANA ===")
print(f"Harga Rumah = {b0:.4f} + ({b1:.4f} × Pendapatan Warga)")
print(f"👉 Interpretasi Bisnis: Setiap kenaikan pendapatan 1 unit ($10.000), harga rumah naik rata-rata {b1*100000:.0f} USD!")

# Visualisasi Garis OLS menembus Titik Sampel
plt.figure(figsize=(8, 5))
sample_idx = np.random.RandomState(42).choice(len(X_test_single), 300, replace=False)
plt.scatter(X_test_single.iloc[sample_idx], y_test.iloc[sample_idx], color='teal', alpha=0.5, label='Data Nyata (Test Sample)')

# Plot garis regresi dengan DataFrame agar tidak muncul peringatan feature names
x_line = pd.DataFrame(
    np.linspace(X_test_single['MedInc'].min(), X_test_single['MedInc'].max(), 100),
    columns=['MedInc']
)
y_line = slr.predict(x_line)
plt.plot(x_line['MedInc'], y_line, color='crimson', lw=3, label='Garis Prediksi OLS (Best Fit Line)')

plt.title("Simple Linear Regression: Pendapatan vs Harga Rumah", fontweight='bold')
plt.xlabel("Median Income (Puluhan Ribu USD)")
plt.ylabel("Median House Value (Ratusan Ribu USD)")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()


## 🌳 4. Melatih Ragam Model Regresi: Linear, Polinomial, hingga Random Forest

Di dunia nyata, harga rumah tidak hanya ditentukan oleh 1 faktor saja. Mari kita latih 3 pendekatan regresi:
1. **Multiple Linear Regression**: Menggunakan seluruh 8 fitur sekaligus.
2. **Polynomial Regression (Degree 2)**: Menangkap hubungan lengkung non-linear dengan menciptakan fitur kuadrat ($X^2$) dan interaksi ($X_1 \cdot X_2$).
3. **Random Forest Regressor**: Pendekatan ensemble non-linear berbasis pohon keputusan yang sangat tangguh di industri data science.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor

# Inisialisasi Model
# 1. Multiple Linear Regression
mlr = LinearRegression()
mlr.fit(X_train_scaled, y_train)

# 2. Polynomial Regression (Degree 2)
poly_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression()
)
fitur_poly = ['MedInc', 'HouseAge', 'AveRooms']
poly_model.fit(X_train[fitur_poly], y_train)

# 3. Random Forest Regressor (Non-linear Ensemble)
rf_reg = RandomForestRegressor(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train)

print("✅ Seluruh model regresi berhasil dilatih dengan sukses!")


## 📏 5. Membedah 3 Metrik Evaluasi Regresi: MAE, RMSE, dan $R^2$

Jangan gunakan Akurasi pada Regresi! Mengapa? Karena kemungkinan model menebak harga rumah Rp 850.231.425,12 dengan presisi desimal 100% tepat adalah nyaris **0%**.  
Kita mengevaluasi regresi dengan mengukur **seberapa dekat tebakan model ke angka kenyataan**:

---

### 1. MAE (*Mean Absolute Error*)
$$\text{MAE} = \frac{1}{n} \sum_{i=1}^n |y_i - \hat{y}_i|$$
* **Analogi**: *"Rata-rata tebakan makelar meleset sebesar Rp 15 juta dari harga asli."*
* **Kelebihan**: Sangat mudah dipahami dan dijelaskan kepada pimpinan/klien non-teknis. Nilainya tidak terlalu terpengaruh oleh outlier ekstrem.

---

### 2. RMSE (*Root Mean Squared Error*)
$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2}$$
* **Analogi**: *"Hukuman denda berlipat ganda untuk tebakan yang meleset sangat jauh."*
* **Kelebihan**: Menghukum kesalahan besar secara kuadratik. Jika Anda membuat kesalahan 10 juta, hukumannya 100; jika salah 50 juta, hukumannya 2500! Sangat cocok jika salah tebak fatal harus dihindari.

---

### 3. $R^2$ (*R-Squared / Koefisien Determinasi*)
Nilai berkisar antara $0.0$ sampai $1.0$ (bisa negatif jika model sangat buruk).
* **Makna Persentase**: Seberapa besar variasi naik-turunnya harga rumah yang **mampu dijelaskan oleh model kita** dibandingkan hanya menebak rata-rata sederhana.
* *Contoh*: Nilai $R^2 = 0.74$ berarti **74% perubahan harga rumah berhasil diterangkan oleh variabel-variabel dalam model**, sedangkan 26% sisanya dipengaruhi faktor lain di luar dataset.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Fungsi pembantu untuk menghitung dan mencetak ketiga metrik
def evaluasi_regresi(nama_model, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {"Model": nama_model, "MAE ($)": mae * 100000, "RMSE ($)": rmse * 100000, "R² Score": r2}

daftar_hasil = []

# Evaluasi Model 1: Multiple Linear Regression
pred_mlr = mlr.predict(X_test_scaled)
daftar_hasil.append(evaluasi_regresi("Multiple Linear Regression", y_test, pred_mlr))

# Evaluasi Model 2: Polynomial Regression
pred_poly = poly_model.predict(X_test[fitur_poly])
daftar_hasil.append(evaluasi_regresi("Polynomial Regression (Deg 2)", y_test, pred_poly))

# Evaluasi Model 3: Random Forest Regressor
pred_rf = rf_reg.predict(X_test)
daftar_hasil.append(evaluasi_regresi("Random Forest Regressor", y_test, pred_rf))

# Tampilkan dalam DataFrame perbandingan
df_evaluasi = pd.DataFrame(daftar_hasil)
print("=== TABEL PERBANDINGAN PERFORMA MODEL REGRESI ===")
print(df_evaluasi.round(3).to_string(index=False))


In [ ]:
# Visualisasi Scatter Plot: Nilai Aktual vs Nilai Prediksi (Random Forest)
plt.figure(figsize=(7, 6))

sample_pts = np.random.RandomState(42).choice(len(y_test), 500, replace=False)
plt.scatter(y_test.iloc[sample_pts] * 100000, pred_rf[sample_pts] * 100000, color='royalblue', alpha=0.5, label='Tebakan Model')

min_val = min(y_test.iloc[sample_pts].min(), pred_rf[sample_pts].min()) * 100000
max_val = max(y_test.iloc[sample_pts].max(), pred_rf[sample_pts].max()) * 100000
plt.plot([min_val, max_val], [min_val, max_val], color='crimson', lw=2.5, linestyle='--', label='Garis Prediksi Sempurna (100% Tepat)')

plt.title("Evaluasi Residual: Harga Aktual vs Tebakan Random Forest", fontweight='bold', fontsize=12)
plt.xlabel("Harga Aktual (USD)")
plt.ylabel("Harga Tebakan Model (USD)")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

print("💡 Cara Membaca Grafik Residual:")
print("1. Semakin rapat titik-titik biru menempel pada garis putus-putus merah, semakin presisi model Anda!")
print("2. Titik di atas garis merah = Model menebak terlalu mahal (Over-estimation).")
print("3. Titik di bawah garis merah = Model menebak terlalu murah (Under-estimation).")


## 🔬 6. Interpretasi Koefisien vs *Feature Importance*

Dalam bisnis, klien tidak hanya ingin tahu berapa taksiran harganya, mereka ingin tahu:  
*"Faktor apa yang paling mendongkrak harga rumah di daerah ini?"*

Mari bandingkan:
1. **Koefisien Regresi Linear**: Menunjukkan pengaruh langsung arah positif (menaikkan harga) atau negatif (menurunkan harga).
2. **Feature Importance Random Forest**: Menunjukkan seberapa sering dan kuat fitur tersebut digunakan pohon dalam membagi data.


In [ ]:
# 1. Koefisien Bobot pada Multiple Linear Regression
koef_df = pd.DataFrame({
    'Fitur': X.columns,
    'Koefisien (Bobot Beta)': mlr.coef_
}).sort_values(by='Koefisien (Bobot Beta)', ascending=False)

# 2. Feature Importance pada Random Forest Regressor
fi_df = pd.DataFrame({
    'Fitur': X.columns,
    'Tingkat Pengaruh (Importance)': rf_reg.feature_importances_
}).sort_values(by='Tingkat Pengaruh (Importance)', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.barplot(data=koef_df, x='Koefisien (Bobot Beta)', y='Fitur', hue='Fitur', palette='coolwarm', legend=False, ax=axes[0])
axes[0].set_title("Koefisien Multiple Linear Regression", fontweight='bold')
axes[0].axvline(0, color='black', lw=1)

sns.barplot(data=fi_df, x='Tingkat Pengaruh (Importance)', y='Fitur', hue='Fitur', palette='viridis', legend=False, ax=axes[1])
axes[1].set_title("Feature Importance Random Forest", fontweight='bold')

plt.tight_layout()
plt.show()

print("Temuan Bisnis Utama:")
print(f"👉 Fitur nomor 1 yang paling dominan mengerek harga rumah adalah: {fi_df.iloc[0]['Fitur']} ({fi_df.iloc[0]['Tingkat Pengaruh (Importance)']*100:.1f}%)!")


## 🧠 7. Kuis Evaluasi Pemahaman Mandiri

Jawablah pertanyaan berikut di benak Anda, lalu jalankan sel kode di bawah untuk mencocokkan logika Anda:

---
* **Soal 1**: Mengapa pada masalah regresi kita TIDAK menggunakan metrik Akurasi (*Accuracy*)?
* **Soal 2**: Kapan Anda lebih memilih metrik **RMSE** dibandingkan **MAE** dalam proyek industri?
* **Soal 3**: Apa arti jika sebuah model regresi menghasilkan nilai **$R^2 = 0.85$**?
* **Soal 4**: Mengapa kita tidak menggunakan parameter `stratify=y` pada `train_test_split` untuk kasus regresi?


In [ ]:
# Kunci jawaban logis evaluasi mandiri Modul 5
kunci_jawaban_modul_5 = {
    "Soal 1": "Akurasi mengukur persentase tebakan kategori yang tepat sama. Pada regresi, target adalah angka kontinu tak berhingga, sehingga peluang tebakan meleset koma desimal sangat tinggi meski tebakannya sudah sangat dekat.",
    "Soal 2": "Gunakan RMSE ketika kesalahan tebak yang bernilai besar ingin dihukum jauh lebih berat (misal: estimasi dosis obat medis atau risiko keuangan kritis yang tidak boleh meleset jauh).",
    "Soal 3": "Artinya 85% variasi perubahan nilai target (harga rumah) berhasil dijelaskan oleh fitur-fitur di dalam model, dan 15% sisanya dipengaruhi faktor eksternal di luar data.",
    "Soal 4": "Karena parameter 'stratify' membutuhkan label kategori diskrit untuk menyeimbangkan kelas. Target regresi berupa angka kontinu, sehingga stratifikasi kategori tidak dapat diterapkan secara langsung."
}

print("=== KUNCI JAWABAN EVALUASI MANDIRI MODUL 05 ===")
for soal, penjelasan in kunci_jawaban_modul_5.items():
    print(f"👉 {soal}:\n   {penjelasan}\n")


## ✍️ 8. Kotak Latihan Bebas (*Playground Challenge*)

### 🎯 Misi Simulasi: Menaksir Harga Properti untuk Calon Pembeli Baru!
Bayangkan Anda menerima calon pembeli properti yang ingin membeli rumah di lingkungan dengan spesifikasi:
* `MedInc` = 6.5 (Pendapatan $65.000)
* `HouseAge` = 25 tahun
* `AveRooms` = 6.0 kamar
* `AveBedrms` = 1.1 kamar tidur
* `Population` = 800 jiwa
* `AveOccup` = 3.0 orang per rumah
* `Latitude` = 34.2
* `Longitude` = -118.3

Tugas Anda:
1. Masukkan profil rumah baru tersebut ke model Random Forest yang telah kita latih.
2. Prediksi berapa perkiraan harga rumah tersebut dalam mata uang Dollar USD!


In [ ]:
# Profil Rumah Baru
rumah_baru = pd.DataFrame([{
    'MedInc': 6.5,
    'HouseAge': 25.0,
    'AveRooms': 6.0,
    'AveBedrms': 1.1,
    'Population': 800.0,
    'AveOccup': 3.0,
    'Latitude': 34.2,
    'Longitude': -118.3
}])

# Prediksi Menggunakan Random Forest Regressor
taksiran_harga = rf_reg.predict(rumah_baru)[0] * 100000

print("=== HASIL TAKSIRAN HARGA PROPERTI (AI PREDICTION) ===")
print(f"Spesifikasi Rumah : Pendapatan Area {rumah_baru['MedInc'].values[0]} | Usia Rumah {rumah_baru['HouseAge'].values[0]} tahun")
print(f"Estimasi Harga    : ${taksiran_harga:,.2f} USD (± Rp {taksiran_harga * 16000:,.0f} IDR) 🏠✨")


## 🎯 9. Checklist Kelulusan Modul 05 (*Hero Checkpoint*)

Beri tanda centang pada hati Anda jika telah menguasai indikator berikut:
- [ ] Saya memahami perbedaan mendasar target **Klasifikasi** (kategori) vs **Regresi** (angka kontinu).
- [ ] Saya memahami filosofi makelar properti: Intercept $\beta_0$ (harga dasar) dan Slope $\beta_1$ (pengaruh per unit).
- [ ] Saya bisa menjelaskan mengapa OLS mencari garis yang meminimalkan *Sum of Squared Residuals*.
- [ ] Saya memahami mengapa metrik Akurasi tidak bisa dipakai pada regresi, dan menguasai **MAE**, **RMSE**, serta nilai **$R^2$**.
- [ ] Saya dapat membaca grafik sebaran residual (Aktual vs Prediksi) dan memahami maknanya.
- [ ] Saya berhasil melakukan inferensi / prediksi angka kontinu pada data baru.

---
🚀 **Selamat! Anda telah menguasai Modul 05 dengan sangat baik!**  
Di modul berikutnya (**Modul 06**), kita akan meninggalkan dunia *Supervised Learning* dan memasuki wilayah **Unsupervised Learning I: Klasterisasi (*Clustering*) & Reduksi Dimensi (PCA)**, di mana mesin akan mencari segmen alami tanpa bantuan kunci jawaban sama sekali!
